# DCA: Datei-zu-Wall Verknuepfung mit CIDOC CRM

Dieses Notebook erstellt in einem ersten Schritt regelbasierte Links zwischen:
- Dateien aus der DROID-TTL
- Wall-Elementen aus der IFC/LBD-TTL

Danach werden die bestaetigten Matches in einer eigenen CIDOC-CRM-TTL exportiert.

Hinweis:
- DROID-TTL enthaelt Dateimetadaten, aber keinen Dateitext.
- Fuer inhaltsbasiertes Matching wird ein CSV mit extrahierter erster Zeile genutzt.

In [ ]:
# Optional: nur falls in der Umgebung nicht vorhanden
!pip install --quiet rdflib pandas

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import re

import pandas as pd
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, XSD

In [ ]:
# ===== Konfiguration (Renku Pfade) =====
rdf_data_dir = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data")
ifc_ttl_path = rdf_data_dir / "FoC_Preserving_IFC-Gantenbein_20260408.ttl"
droid_ttl_path = rdf_data_dir / "036_WeingutGantenbein_DROID.ttl"

# CSV mit Inhaltsextrakt der Produktionsdateien (mind. first_line + file_name oder file_uri)
# Erwartete Spalten (flexibel): file_name, first_line, file_uri
content_extract_csv = rdf_data_dir / "gantenbein_file_content_extract.csv"

# Ausgaben
report_csv_path = rdf_data_dir / "Gantenbein_wall_file_matches_report.csv"
crm_ttl_output = rdf_data_dir / "Gantenbein_CRM_WallFileLinks.ttl"

for p in [ifc_ttl_path, droid_ttl_path]:
    if not p.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {p}")

print("Konfiguration OK")
print(f"IFC TTL:   {ifc_ttl_path}")
print(f"DROID TTL: {droid_ttl_path}")
print(f"CSV:       {content_extract_csv} (optional, aber empfohlen)")

In [ ]:
# ===== Graphen laden =====
ifc_g = Graph()
ifc_g.parse(ifc_ttl_path, format="turtle")

droid_g = Graph()
droid_g.parse(droid_ttl_path, format="turtle")

print(f"IFC-Triples:   {len(ifc_g):,}")
print(f"DROID-Triples: {len(droid_g):,}")

In [ ]:
# ===== Namespaces =====
BOT = Namespace("https://w3id.org/bot#")
PROPS = Namespace("http://lbd.arch.rwth-aachen.de/props#")
DCTERMS = Namespace("http://purl.org/dc/terms/")
PREMIS = Namespace("http://www.loc.gov/premis/rdf/v3/")
DCA_ID = Namespace("http://dca.ethz.ch/id/")
DCA = Namespace("http://dca.example.org/")
CRM = Namespace("http://www.cidoc-crm.org/cidoc-crm/")
CRMDIG = Namespace("http://www.ics.forth.gr/isl/CRMdig/")

In [ ]:
# ===== Hilfsfunktionen =====
def normalize_token(text: str) -> str:
    t = text.strip().lower()
    t = re.sub(r"particles$", "", t)
    t = re.sub(r"[^a-z0-9]+", "_", t)
    t = re.sub(r"_+", "_", t).strip("_")
    return t

def extract_wall_token(first_line: str) -> str | None:
    if not isinstance(first_line, str):
        return None
    # Beispiel: DEFDAT Wall_C3_1_3Particles
    m = re.search(r"(wall[\-_a-zA-Z0-9]+)", first_line, flags=re.IGNORECASE)
    if not m:
        return None
    raw = m.group(1)
    raw = re.sub(r"particles$", "", raw, flags=re.IGNORECASE)
    return raw

In [ ]:
# ===== Wall-Kandidaten aus IFC/LBD extrahieren =====
wall_rows = []

for s in ifc_g.subjects(RDF.type, BOT.Element):
    uri_txt = str(s)
    uri_local = uri_txt.rsplit("/", 1)[-1].lower()

    labels = [str(o) for o in ifc_g.objects(s, RDFS.label)]
    global_ids = [str(o) for o in ifc_g.objects(s, PROPS.globalIdIfcRoot_attribute_simple)]

    is_wall_uri = "wall_" in uri_local
    is_wall_label = any("wall" in lb.lower() for lb in labels)

    if not (is_wall_uri or is_wall_label):
        continue

    canonical_label = labels[0] if labels else uri_txt.rsplit("/", 1)[-1]
    wall_rows.append({
        "ifc_wall_uri": uri_txt,
        "ifc_wall_label": canonical_label,
        "ifc_wall_label_norm": normalize_token(canonical_label),
        "ifc_wall_uri_norm": normalize_token(uri_txt.rsplit("/", 1)[-1]),
        "ifc_global_id": global_ids[0] if global_ids else None,
    })

walls_df = pd.DataFrame(wall_rows).drop_duplicates(subset=["ifc_wall_uri"])
print(f"Wall-Kandidaten gefunden: {len(walls_df)}")
walls_df.head(10)

In [ ]:
# ===== DROID-Dateien extrahieren =====
file_rows = []

for s in droid_g.subjects(DCTERMS.title, None):
    title_vals = [str(o) for o in droid_g.objects(s, DCTERMS.title)]
    id_vals = [str(o) for o in droid_g.objects(s, DCTERMS.identifier)]
    fmt_vals = [str(o) for o in droid_g.objects(s, PREMIS.hasFormatName)]

    title = title_vals[0] if title_vals else None
    file_rows.append({
        "file_uri": str(s),
        "file_name": title,
        "file_name_norm": normalize_token(title) if title else None,
        "file_identifier": id_vals[0] if id_vals else None,
        "file_format": fmt_vals[0] if fmt_vals else None,
    })

droid_files_df = pd.DataFrame(file_rows).drop_duplicates(subset=["file_uri"])
print(f"DROID-Dateien gefunden: {len(droid_files_df)}")
droid_files_df.head(10)

In [ ]:
# ===== Inhaltsextrakt laden (CSV) oder Fallback-Beispiel =====
if content_extract_csv.exists():
    content_df = pd.read_csv(content_extract_csv)
    print(f"CSV geladen: {content_extract_csv}")
else:
    print("WARNUNG: CSV nicht gefunden. Nutze Demo-Datensatz fuer ersten Testlauf.")
    content_df = pd.DataFrame([
        {
            "file_name": "pos54R_Brick.dat",
            "first_line": "DEFDAT Wall_C3_1_3Particles",
            "file_uri": None,
        }
    ])

# Flexible Spaltenbehandlung
if "file_name" not in content_df.columns:
    content_df["file_name"] = None
if "first_line" not in content_df.columns:
    raise ValueError("CSV benoetigt mindestens die Spalte 'first_line'.")
if "file_uri" not in content_df.columns:
    content_df["file_uri"] = None

content_df["file_name_norm"] = content_df["file_name"].fillna("").map(normalize_token)
content_df["wall_token_raw"] = content_df["first_line"].map(extract_wall_token)
content_df["wall_token_norm"] = content_df["wall_token_raw"].fillna("").map(normalize_token)

content_df.head(20)

In [ ]:
# ===== Matching: Datei -> IFC-Wall =====
matches = []

# Zur Dateiidentifikation bevorzugt file_uri, ansonsten file_name
for _, row in content_df.iterrows():
    token_norm = row.get("wall_token_norm", "")

    # Kandidaten aus Wall-Label und URI-Namen
    cands = walls_df[
        (walls_df["ifc_wall_label_norm"] == token_norm)
        | (walls_df["ifc_wall_uri_norm"].str.contains(token_norm, na=False))
        | (pd.Series([token_norm]).str.contains("wall").iloc[0] and walls_df["ifc_wall_label_norm"].str.contains(token_norm, na=False))
    ]

    # Datei auf DROID aufloesen
    droid_subset = pd.DataFrame()
    if pd.notna(row.get("file_uri")) and row.get("file_uri"):
        droid_subset = droid_files_df[droid_files_df["file_uri"] == str(row["file_uri"])]
    if droid_subset.empty and pd.notna(row.get("file_name")):
        fn = normalize_token(str(row.get("file_name", "")))
        droid_subset = droid_files_df[droid_files_df["file_name_norm"] == fn]

    file_uri = droid_subset.iloc[0]["file_uri"] if not droid_subset.empty else None
    file_name = droid_subset.iloc[0]["file_name"] if not droid_subset.empty else row.get("file_name")

    if len(cands) == 1:
        status = "exact_or_normalized"
        confidence = 0.95
        wall_uri = cands.iloc[0]["ifc_wall_uri"]
        wall_label = cands.iloc[0]["ifc_wall_label"]
    elif len(cands) > 1:
        status = "ambiguous"
        confidence = 0.4
        wall_uri = None
        wall_label = None
    else:
        status = "no_match"
        confidence = 0.0
        wall_uri = None
        wall_label = None

    matches.append({
        "file_uri": file_uri,
        "file_name": file_name,
        "first_line": row.get("first_line"),
        "wall_token_raw": row.get("wall_token_raw"),
        "wall_token_norm": token_norm,
        "match_status": status,
        "match_confidence": confidence,
        "ifc_wall_uri": wall_uri,
        "ifc_wall_label": wall_label,
        "candidate_count": int(len(cands)),
    })

matches_df = pd.DataFrame(matches)
matches_df.to_csv(report_csv_path, index=False)
print(f"Matching-Report gespeichert: {report_csv_path}")
matches_df

In [ ]:
# ===== CIDOC CRM / CRMdig Graph erzeugen =====
crm_g = Graph()
crm_g.bind("crm", CRM)
crm_g.bind("crmdig", CRMDIG)
crm_g.bind("rdfs", RDFS)
crm_g.bind("dca", DCA)
crm_g.bind("bot", BOT)

# Klassen
E13 = URIRef(CRM + "E13_Attribute_Assignment")
E73 = URIRef(CRM + "E73_Information_Object")
E55 = URIRef(CRM + "E55_Type")
E52 = URIRef(CRM + "E52_Time-Span")
E28 = URIRef(CRM + "E28_Conceptual_Object")
D1 = URIRef(CRMDIG + "D1_Digital_Object")

# Pradikate
P138 = URIRef(CRM + "P138_represents")
P140 = URIRef(CRM + "P140_assigned_attribute_to")
P141 = URIRef(CRM + "P141_assigned")
P177 = URIRef(CRM + "P177_assigned_property_of_type")
P3 = URIRef(CRM + "P3_has_note")
P4 = URIRef(CRM + "P4_has_time-span")
P82A = URIRef(CRM + "P82a_begin_of_the_begin")

# Typ fuer die Matching-Aussage
match_type_uri = URIRef(DCA + "type_production_file_matches_conceptual_wall")
crm_g.add((match_type_uri, RDF.type, E55))
crm_g.add((match_type_uri, RDFS.label, Literal("Production file matches conceptual wall")))

# Laufzeit als Time-Span
run_ts = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
timespan_uri = URIRef(DCA + f"timespan_match_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
crm_g.add((timespan_uri, RDF.type, E52))
crm_g.add((timespan_uri, P82A, Literal(run_ts, datatype=XSD.dateTime)))

accepted = matches_df[matches_df["match_status"] == "exact_or_normalized"].copy()

def make_local_id(txt: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", txt).strip("_").lower()

for i, row in accepted.reset_index(drop=True).iterrows():
    src_file_uri_txt = row.get("file_uri")
    src_ifc_wall_uri_txt = row.get("ifc_wall_uri")

    if not isinstance(src_ifc_wall_uri_txt, str):
        continue

    token = row.get("wall_token_norm") or f"wall_{i+1}"
    token = make_local_id(str(token))

    # 1) Konzeptuelle Wand (unabhangig von Reprasentationen)
    conceptual_wall_uri = URIRef(DCA + f"conceptual_wall_{token}")
    crm_g.add((conceptual_wall_uri, RDF.type, E28))
    crm_g.add((conceptual_wall_uri, RDFS.label, Literal(f"Conceptual wall {row.get('wall_token_raw') or token}")))

    # 2) IFC-Wandreprasentation als eigenes digitales Objekt
    ifc_repr_uri = URIRef(DCA + f"ifc_wall_representation_{token}")
    crm_g.add((ifc_repr_uri, RDF.type, E73))
    crm_g.add((ifc_repr_uri, RDF.type, D1))
    crm_g.add((ifc_repr_uri, RDFS.label, Literal(f"IFC wall representation {token}")))
    crm_g.add((ifc_repr_uri, P138, conceptual_wall_uri))
    crm_g.add((ifc_repr_uri, P3, Literal(f"Source IFC wall URI: {src_ifc_wall_uri_txt}")))

    if isinstance(row.get("ifc_wall_label"), str):
        crm_g.add((ifc_repr_uri, P3, Literal(f"IFC wall label: {row.get('ifc_wall_label')}")))

    # 3) Produktionsdatei als eigenes digitales Objekt (CRMdig)
    file_key = make_local_id(str(row.get("file_name") or src_file_uri_txt or f"file_{i+1}"))
    prod_file_uri = URIRef(DCA + f"production_file_{file_key}")
    crm_g.add((prod_file_uri, RDF.type, E73))
    crm_g.add((prod_file_uri, RDF.type, D1))
    crm_g.add((prod_file_uri, RDFS.label, Literal(str(row.get("file_name") or file_key))))
    crm_g.add((prod_file_uri, P138, conceptual_wall_uri))

    if isinstance(src_file_uri_txt, str):
        crm_g.add((prod_file_uri, P3, Literal(f"Source DROID file URI: {src_file_uri_txt}")))

    # Match-Aussage als einfaches nachvollziehbares Assignment
    assign_uri = URIRef(DCA + f"assign_prodfile_to_conceptual_wall_{i+1}")
    crm_g.add((assign_uri, RDF.type, E13))
    crm_g.add((assign_uri, P140, prod_file_uri))
    crm_g.add((assign_uri, P141, conceptual_wall_uri))
    crm_g.add((assign_uri, P177, match_type_uri))
    crm_g.add((assign_uri, P4, timespan_uri))

    note = (
        f"Matched by notebook rule from first line: {row.get('first_line')} ; "
        f"token={row.get('wall_token_raw')} ; confidence={row.get('match_confidence')}"
    )
    crm_g.add((assign_uri, P3, Literal(note)))

print(f"CRM-Triples erzeugt: {len(crm_g):,}")
print(f"Akzeptierte Matches: {len(accepted)}")

In [ ]:
# ===== Export =====
crm_g.serialize(destination=crm_ttl_output, format="turtle")
print(f"CRM TTL gespeichert: {crm_ttl_output}")

print("\nMatch-Status Uebersicht:")
print(matches_df["match_status"].value_counts(dropna=False))

print("\nTop 20 Matches:")
matches_df.head(20)

## Nächste Schritte

1. Vollstaendige IFC-TTL eintragen (ifc_ttl_path).
2. CSV mit Dateitext-Extrakt erzeugen und als `gantenbein_file_content_extract.csv` ablegen.
3. `ambiguous`-Fälle manuell reviewen und ggf. Mapping-Tabelle pflegen.
4. Danach optional E12 Production-Ereignisse pro bestaetigter Verknuepfung ergaenzen.